# TabPFN drift - real datasets (Bike Sharing, Appliances Energy, Beijing PM2.5)
Runs the six **real TabPFN** context policies on the three real datasets, in stages.
Outputs go to `/content/results_real` (no Google Drive). **After every stage, run the download
cell** and keep the zip: if Colab disconnects, `/content` is wiped. To continue after a disconnect,
run the setup cells again, then the *restore* cell with your latest zip; finished CSVs are skipped.

Real data has **no known drift point**: we look at error over time (rolling RMSE), not recovery time.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Upload the code zip (includes `data/raw`, which the loaders need)
Make it on your laptop in PowerShell, inside the repo folder (skips `.venv`, includes `data/raw`).
Make it **after** the local baseline runs have finished, so `results/` has the baselines for comparison:
```
tar.exe -a -cf ..\tabpfn-drift.zip --exclude=__pycache__ common data eval methods results notebooks run_experiments.py smoke_test.py time_tabpfn.py plot_check.py requirements.txt
```
Then pick `tabpfn-drift.zip` (one folder up from the repo) when the upload box appears.

In [ ]:
from google.colab import files
files.upload()
!rm -rf /content/tabpfn-drift && mkdir -p /content/tabpfn-drift
!unzip -o -q tabpfn-drift.zip -d /content/tabpfn-drift
%cd /content/tabpfn-drift
!ls data/raw

## 3. Install packages (numpy, pandas, sklearn, torch are already on Colab)

In [ ]:
!pip install -q tabpfn river

## 4. TabPFN token (never paste it into a code cell or commit it)
Route A: licence key from https://ux.priorlabs.ai -> `TABPFN_TOKEN`.
Route B: accept the TabPFN model terms on huggingface.co, make a *Read* token -> `HF_TOKEN`.
Press Enter to skip either one.

In [ ]:
import os, getpass
t = getpass.getpass('TABPFN_TOKEN (Enter to skip): ').strip()
if t: os.environ['TABPFN_TOKEN'] = t
h = getpass.getpass('HF_TOKEN (Enter to skip): ').strip()
if h: os.environ['HF_TOKEN'] = h

## 5. Settings + download helper (used after every stage)

In [ ]:
import os, shutil
from google.colab import files
OUT = '/content/results_real'
os.makedirs(OUT, exist_ok=True)
CMAX = 2000
POLICIES = 'vanilla sliding_window recency adwin knn validation'   # validation last: it is the slowest

def download_results(stage):
    """Zip everything in OUT and download it to your laptop."""
    n = len([f for f in os.listdir(OUT) if f.endswith('.csv')])
    path = shutil.make_archive(f'/content/results_real_{stage}', 'zip', '/content', 'results_real')
    print(f'{n} result CSVs -> {path}')
    files.download(path)

## 6. (Only after a disconnect) restore your last downloaded `results_real_*.zip`

In [ ]:
for name in files.upload():
    shutil.unpack_archive(name, '/content')
print(sorted(os.listdir(OUT)))

## 7. Pipeline self-test (stand-in model, not TabPFN). Must end with ALL SMOKE TESTS PASSED and show 3 real loaders OK

In [ ]:
%cd /content/tabpfn-drift
!python smoke_test.py 2>&1 | tail -8

## 8. Time TabPFN (also proves the token works). Note the seconds for 2000 rows.

In [ ]:
!python time_tabpfn.py

## 9. Rough time estimate per stage
Based on the synthetic runs (10 features): vanilla ~115 s and validation ~700 s per 59 blocks.
The other four policies are assumed to cost about the same as vanilla. Real datasets have
11-26 features, so real times may be higher. If cell 8 shows a different time for 2000 rows,
put it in `VANILLA_S` and re-run this cell.

In [ ]:
from data.real import load_real
VANILLA_S = 115 / 59          # seconds per block, vanilla (cell 8's 2000-row number, if you have it)
VALIDATION_S = 700 / 59       # seconds per block, validation (about 6x vanilla)
blocks = {d: load_real(d).df.block.nunique() - 1 for d in ['bike', 'energy', 'pm25']}   # scored blocks

def minutes(ds, seeds=1, validation=True):
    per_block = 5 * VANILLA_S + (VALIDATION_S if validation else 0)
    return blocks[ds] * per_block * seeds / 60

for stage, ds in [('A', 'bike'), ('B', 'energy'), ('C', 'pm25')]:
    print(f"Stage {stage} {ds:6s} ({blocks[ds]} blocks): ~{minutes(ds):.0f} min "
          f"(validation alone ~{blocks[ds] * VALIDATION_S / 60:.0f} min)")
d_all = sum(minutes(ds, seeds=2) for ds in blocks)
d_noval = sum(minutes(ds, seeds=2, validation=False) for ds in blocks)
print(f"Stage D seeds 1+2, all datasets: ~{d_all / 60:.1f} h with validation, ~{d_noval / 60:.1f} h without")
if blocks['pm25'] * VALIDATION_S / 60 > 30:
    print("WARNING: pm25 validation alone takes over 30 min per seed. It runs last in stage C, so you "
          "can stop it and still keep the other five. In stage D, consider skipping validation.")

## Stage A - Bike Sharing, seed 0, all six policies
Predicts hourly bike rentals block by block (200 hours per block), each policy choosing its own
context rows. Resumable: finished CSVs are skipped.

In [ ]:
!python run_experiments.py --dataset bike --seeds 0 --methods {POLICIES} --c-max {CMAX} --out {OUT}

In [ ]:
download_results('A_bike')

### Rolling RMSE on Bike Sharing (seed 0)
RMSE over the last 5 blocks, per policy (solid). Local baselines from the zip are dashed, if present.
Triangles = ADWIN drift flags. There is **no "true drift" line**: on real data it is unknown.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from plot_check import read
from methods.policies import POLICIES as POLICY_NAMES
W = 5   # blocks; all blocks have 200 rows, so this is the exact RMSE over the last 1000 rows
def rolling(d): return np.sqrt((d.rmse ** 2).rolling(W).mean())

fig, ax = plt.subplots(figsize=(11, 5))
rows = []
for i, m in enumerate(POLICY_NAMES):
    d = read(OUT, 'bike', 0, m)
    if d is None: continue
    r = rolling(d)
    ax.plot(r.index, r, color=f'C{i}', label=m)
    fl = d.index[d.drift_flag == 1]
    ax.scatter(fl, r.reindex(fl), marker='v', s=60, color=f'C{i}')
    rows.append((m, 'TabPFN policy', d.rmse.mean()))
for j, m in enumerate(['XGBoost', 'MLP', 'AdaptiveForest']):
    d = read('/content/tabpfn-drift/results', 'bike', 0, m)
    if d is None: continue
    r = rolling(d)
    ax.plot(r.index, r, color=f'C{6 + j}', ls='--', alpha=0.8, label=f'{m} (baseline)')
    rows.append((m, 'baseline (local run)', d.rmse.mean()))
ax.set(xlabel='block (200 hours each)', ylabel=f'rolling RMSE over {W} blocks (bikes per hour)',
       title='Bike Sharing, seed 0 - real TabPFN policies vs baselines')
ax.legend(fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(f'{OUT}/rolling_bike_0.png', dpi=120); plt.show()
import pandas as pd
print(pd.DataFrame(rows, columns=['method', 'kind', 'mean RMSE']).sort_values('mean RMSE').round(2).to_string(index=False))

## Stage B - Appliances Energy, seed 0, all six policies
Predicts appliance energy use (10-minute steps, 200 steps = about 33 hours per block).

In [ ]:
!python run_experiments.py --dataset energy --seeds 0 --methods {POLICIES} --c-max {CMAX} --out {OUT}

In [ ]:
download_results('B_energy')

## Stage C - Beijing PM2.5, seed 0, all six policies (the longest dataset: 207 blocks)
Predicts hourly air pollution from weather. Validation runs last and is the slowest part
(see cell 9). If time is short, stop the cell during validation: the other five are already saved.

In [ ]:
!python run_experiments.py --dataset pm25 --seeds 0 --methods {POLICIES} --c-max {CMAX} --out {OUT}

In [ ]:
download_results('C_pm25')

## Stage D (optional) - seeds 1 and 2 for all three datasets
The data is the same; the seed changes TabPFN's randomness and the random context samples.
This is long (see cell 9: hours with validation). Recommended: skip validation here by
switching to the second `D_METHODS` line. It downloads a zip after each dataset.

In [ ]:
D_METHODS = POLICIES                                        # all six (slow)
# D_METHODS = 'vanilla sliding_window recency adwin knn'    # recommended: without validation
for ds in ['bike', 'energy', 'pm25']:
    get_ipython().system(f'python run_experiments.py --dataset {ds} --seeds 1 2 --methods {D_METHODS} '
                         f'--c-max {CMAX} --out {OUT}')
    download_results(f'D_{ds}')

**When you are done:** keep the downloaded zips. Unzip the latest one into a new folder in the repo
(e.g. `results_real_colab/`), not over `results/`, and the next step will merge and report it.